# Tacotron2-VAE — Treinamento Simplificado

Demonstração do pipeline Tacotron2-VAE usando dados de `src/data/loader_TTS_GST`.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import torch
import matplotlib.pyplot as plt

os.chdir(Path.cwd().parent)
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "training" / "training-tacotron2-vae"))

from models.tacotron2_vae.hparams import create_hparams
from models.tacotron2_vae.model import load_tacotron2_vae_model, get_model_size_info
from losses import Tacotron2LossVAE
from text_processing import TextProcessor
from utils import ARTIFACTS_DIR, TextMelCollate, create_dataset, create_dataloader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# 1) Preprocess (gera filelists + symbols a partir do loader_TTS_GST)
import subprocess

preprocess_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "training" / "training-tacotron2-vae" / "preprocess.py"),
]
subprocess.run(preprocess_cmd, check=True)

summary_path = ARTIFACTS_DIR / "preprocess_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

In [ ]:
# 2) Construir modelo
text_processor = TextProcessor.load(ARTIFACTS_DIR / "symbols.json")
hparams = create_hparams({
    "n_symbols": text_processor.n_symbols,
    "batch_size": 4,
    "anneal_function": "constant",
})

model = load_tacotron2_vae_model(hparams, device=device)
criterion = Tacotron2LossVAE(hparams)
optimizer = torch.optim.Adam(model.parameters(), lr=hparams.learning_rate, weight_decay=hparams.weight_decay)

print(get_model_size_info(model))

In [ ]:
# 3) Treinamento simplificado
train_dataset = create_dataset(ARTIFACTS_DIR / "train.csv", text_processor)
collate_fn = TextMelCollate(hparams.n_frames_per_step)
train_loader = create_dataloader(train_dataset, batch_size=hparams.batch_size, num_workers=0, collate_fn=collate_fn, shuffle=True)

loss_history = []
recon_history = []
kl_history = []
max_steps = 20
iteration = 0

model.train()
for batch in train_loader:
    if iteration >= max_steps:
        break

    optimizer.zero_grad()
    x, y = model.parse_batch(batch, device)
    y_pred = model((x[0], x[1], x[2], x[3], x[4], x[5], x[6]))
    loss, recon_loss, kl_loss, kl_weight = criterion(y_pred, y, iteration)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), hparams.grad_clip_thresh)
    optimizer.step()

    loss_history.append(loss.item())
    recon_history.append(recon_loss.item())
    kl_history.append(kl_loss.item())
    print(f"step={iteration} loss={loss.item():.4f} recon={recon_loss.item():.4f} kl={kl_loss.item():.4f} kl_w={kl_weight:.6f}")
    iteration += 1

In [ ]:
# 4) Resultados
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(loss_history)
ax[0].set_title("Total Loss")
ax[1].plot(recon_history)
ax[1].set_title("Reconstruction Loss")
ax[2].plot(kl_history)
ax[2].set_title("KL Loss")
plt.tight_layout()
plt.show()

In [ ]:
# 5) Inferência simplificada (mel previsto vs alvo no batch)
model.eval()
with torch.no_grad():
    batch = next(iter(train_loader))
    x, y = model.parse_batch(batch, device)
    y_pred = model((x[0], x[1], x[2], x[3], x[4], x[5], x[6]))
    pred_mel = y_pred[1][0].cpu()
    target_mel = y[0][0].cpu()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(target_mel, aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("Target Mel")
axes[1].imshow(pred_mel, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("Predicted Mel (postnet)")
plt.tight_layout()
plt.show()